# Feature engineering memo — EURUSD 4H

**Author:** Chertock  
**Audience:** senior quant researcher  
**Data:** `EURUSD_4H_MetaQuotes_Demo_READY.csv` (4H OHLCV + `high_time` / `low_time` from `HighLowTime.py` using **M1** drill-down)

This notebook follows the course **Feature Analysis Pipeline** (*Alpha Quant Notes — Data Preprocessing & Feature Analysis*): engineer features → define forward log returns at several horizons → Spearman **IC** → rolling IC stability → multicollinearity → distribution hygiene → final checklist.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import spearmanr, zscore

ROOT = Path.cwd().resolve()
if not (ROOT / "Quantreo").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from Quantreo.DataPreprocessing import ema, rsi

DATA_PATH = ROOT / "Data/FixTimeBars/EURUSD_4H_MetaQuotes_Demo_READY.csv"
sns.set_theme(style="whitegrid", context="notebook")

## Prerequisites: HighLowTime

Backtests need **when** the high and low occurred inside each bar (see course notes: TP/SL ordering). We confirm those columns exist before adding signal features.

**Garbage rows:** `HighLowTime.py` reports the share of bars where high/low timestamps tie; ours was **< 1%** on M1 drill-down (course target).

In [ ]:
df = pd.read_csv(DATA_PATH, index_col="time", parse_dates=True).sort_index()
for col in ("high_time", "low_time"):
    df[col] = pd.to_datetime(df[col])

assert {"high_time", "low_time"}.issubset(df.columns)
print(f"Bars: {len(df)}  |  {df.index.min()} → {df.index.max()}")
df[["open", "high", "low", "close", "high_time", "low_time"]].head(2)

## Step 1 — Engineer features

We use `Quantreo/DataPreprocessing.py` for RSI 14 and EMA 20. The earlier screen had two near-duplicate pairs (`SMA_20` with `EMA_20`, `RSI` with `price_vs_ema20`, both Spearman ρ ≈ 0.99). This pass **keeps RSI and EMA 20**, drops those twins, and also drops `SMA_50`, `SMA_diff`, and `atr_pct`.

**VWAP** is the session price level. Typical price `(high + low + close) / 3`, weighted by `tick_volume` (`real_volume` is 0 on this file), reset each calendar day. This file has six 4H bars per day.

No predictive judgment yet — only new columns.

In [ ]:
df = rsi(df, "close", 14)
df = ema(df, "close", 20)

typical = (df["high"] + df["low"] + df["close"]) / 3.0
volume = df["tick_volume"].astype(float)
session = df.index.normalize()
cum_vol = volume.groupby(session).cumsum()
df["VWAP"] = (typical * volume).groupby(session).cumsum() / cum_vol.replace(0, np.nan)

FEATURES = [
    "RSI",
    "EMA_20",
    "VWAP",
]
df[FEATURES].tail(3)

## Step 2 — Forward log returns

Research-only labels (dropped before live/backtest features). One column per holding period:

`log_return_{label} = log(close / close.shift(h)).shift(-h)`

Bars are trading bars: this file prints six 4H bars per session, and the weekend is a timestamp gap, not extra rows.

| Column | Bars ahead | Holding period |
|--------|------------|----------------|
| `log_return_4h` | 1 | 4 hours |
| `log_return_1d` | 6 | 1 session |
| `log_return_1w` | 30 | 5 sessions |
| `log_return_1m` | 126 | 21 sessions |

Steps 4–6 stay on **`log_return_4h`**. The IC table compares all four horizons so a feature can show a different return profile at a longer hold.

In [ ]:
HORIZONS = {
    "4h": 1,
    "1d": 6,
    "1w": 30,
    "1m": 126,
}
PRIMARY_RETURN = "log_return_4h"
LOG_RETURNS = [f"log_return_{label}" for label in HORIZONS]

for label, h in HORIZONS.items():
    df[f"log_return_{label}"] = np.log(df["close"] / df["close"].shift(h)).shift(-h)

work = df.dropna(subset=[PRIMARY_RETURN] + FEATURES).copy()
print(f"Rows for primary screen ({PRIMARY_RETURN}): {len(work)}")
print("Non-null rows by horizon:")
print(df[LOG_RETURNS].notna().sum())
df[LOG_RETURNS].describe()

## Step 3 — Individual predictive power (Information Coefficient)

Spearman rank correlation between each feature and a forward log return = **IC**.

The table and heatmap report IC at **4h, 1d, 1w, and 1m**. **`pass_IC` uses `log_return_4h` only**. Each horizon is scored on the rows where that return and every feature are present, so a longer hold does not shrink the 4h sample.

**Interpretation (notes):**
- |IC| > 0.05 — worth investigating  
- |IC| > 0.10 — meaningful  
- p < 0.05 — statistically significant  

`pass_IC` is a **log**, not a filter. A feature that misses the threshold stays in the set. One column is not supposed to be a complete signal on its own.

In [ ]:
ic_parts = []
for label, col in zip(HORIZONS, LOG_RETURNS):
    sample = df.dropna(subset=[col] + FEATURES)
    rows = []
    for feature in FEATURES:
        ic, pvalue = spearmanr(sample[feature], sample[col])
        rows.append({"feature": feature, f"IC_{label}": ic, f"p_{label}": pvalue})
    part = pd.DataFrame(rows).set_index("feature")
    part[f"n_{label}"] = len(sample)
    ic_parts.append(part)

ic_by_horizon = pd.concat(ic_parts, axis=1)
ic_table = ic_by_horizon.reset_index()
ic_table["IC"] = ic_table["IC_4h"]
ic_table["p_value"] = ic_table["p_4h"]
ic_table["abs_IC"] = ic_table["IC"].abs()
ic_table["pass_IC"] = (ic_table["abs_IC"] > 0.05) & (ic_table["p_value"] < 0.05)
ic_table = ic_table.sort_values("abs_IC", ascending=False)

ic_matrix = ic_table.set_index("feature")[[f"IC_{label}" for label in HORIZONS]]
ic_matrix.columns = list(HORIZONS)
plt.figure(figsize=(6.5, 4))
sns.heatmap(ic_matrix, annot=True, fmt=".3f", cmap="coolwarm", center=0)
plt.title("Spearman IC by forward log-return horizon")
plt.tight_layout()
plt.show()

print("Rows used at each horizon:")
print({label: int(ic_table[f"n_{label}"].iloc[0]) for label in HORIZONS})
ic_table[["feature", "IC_4h", "IC_1d", "IC_1w", "IC_1m", "p_4h", "pass_IC"]]

### Step 3 — Findings (edit after run)

The set is **RSI**, **EMA 20**, and session **VWAP**. `SMA_20` and `price_vs_ema20` were removed as the redundant twins of EMA 20 and RSI. `SMA_50`, `SMA_diff`, and `atr_pct` were removed with them.

- **`EMA_20` and `VWAP`:** Both are price levels. If their correlation is still extreme, the level is the same object twice; the difference is the session anchor, not a second signal.
- **`RSI`:** Classic mean-reversion/momentum tension; IC magnitude depends on horizon and sample.
- **Horizon profile:** Compare `IC_4h`, `IC_1d`, `IC_1w`, and `IC_1m` before treating a 4h pass as a longer-hold signal. Sign or magnitude that flips across columns is a different return profile, not a second copy of the same edge.

On a **~200-bar demo sample**, p-values are unstable. The notes’ thresholds are still recorded as flags. They do not remove features. **`log_return_1m`** looks 126 bars ahead, so after indicator warmup only a few dozen rows remain — read `IC_1m` as illustrative.

In [ ]:
CANDIDATES = list(FEATURES)
failed_ic = sorted(ic_table.loc[~ic_table["pass_IC"], "feature"])
print("IC below 4h threshold (kept):", failed_ic)
print("Features carried forward:", CANDIDATES)

## Step 4 — Stability over time (rolling IC)

Notes use `window=252` bars; our demo history is shorter, so we use **`window=60`** and treat results as **illustrative**. Rolling IC is versus **`log_return_4h` only**. We report **Mean IC**, **IC Std**, and **IC IR** = Mean / Std.

`pass_stability` (IC IR > 0.5) is logged. Every feature continues to the correlation and distribution checks.

In [ ]:
window = min(60, len(work) // 3)
stability = []

fig, axes = plt.subplots(len(CANDIDATES), 1, figsize=(12, 3 * max(len(CANDIDATES), 1)), sharex=True)
if len(CANDIDATES) == 1:
    axes = [axes]

for ax, feature in zip(axes, CANDIDATES):
    rolling_ic = work[[feature, PRIMARY_RETURN]].rolling(window).corr().unstack()[PRIMARY_RETURN][feature]
    rolling_ic.plot(ax=ax, title=f"Rolling IC — {feature} (window={window})")
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    mean_ic = rolling_ic.mean()
    std_ic = rolling_ic.std()
    ic_ir = mean_ic / std_ic if std_ic and not np.isnan(std_ic) else np.nan
    stability.append({
        "feature": feature,
        "mean_IC": mean_ic,
        "IC_std": std_ic,
        "IC_IR": ic_ir,
        "pass_stability": (not np.isnan(ic_ir)) and abs(ic_ir) > 0.5,
    })

plt.tight_layout()
plt.show()

stability_df = pd.DataFrame(stability)
stability_df

In [ ]:
SELECTED = list(CANDIDATES)
failed_stability = sorted(stability_df.loc[~stability_df["pass_stability"], "feature"])
print("Stability below threshold (kept):", failed_stability)
print("Features carried forward:", SELECTED)

## Step 5 — Multicollinearity (Spearman between features)

**Notes:** |ρ| > 0.7 means the pair is likely redundant. We **log the pair** and keep both features. IC vs `log_return_4h` is already in the table above if you later want to prefer one of them.

In [ ]:
corr_matrix = work[SELECTED].corr(method="spearman")

plt.figure(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Feature correlation matrix (Spearman)")
plt.tight_layout()
plt.show()

REDUNDANT = 0.7
redundant_pairs = []
for i, a in enumerate(SELECTED):
    for b in SELECTED[i + 1 :]:
        rho = corr_matrix.loc[a, b]
        if abs(rho) >= REDUNDANT:
            redundant_pairs.append({"feature_a": a, "feature_b": b, "rho": rho})
            print(f"Redundant (kept both): {a} vs {b} (rho={rho:.2f})")

FINAL_FEATURES = list(SELECTED)
if not redundant_pairs:
    print("No pair with |rho| >= 0.7")
print("Features carried forward:", FINAL_FEATURES)

## Step 6 — Feature distributions

Skew/outliers distort correlation. Notes: flag **> 1%** of points beyond 3σ. `pass_dist` is logged. Optional winsorize / z-score can wait until modeling.

In [ ]:
fig, axes = plt.subplots(1, len(FINAL_FEATURES), figsize=(4 * len(FINAL_FEATURES), 3))
if len(FINAL_FEATURES) == 1:
    axes = [axes]
for ax, feature in zip(axes, FINAL_FEATURES):
    work[feature].hist(bins=30, ax=ax)
    ax.set_title(feature)
plt.tight_layout()
plt.show()

dist_rows = []
for feature in FINAL_FEATURES:
    z = np.abs(zscore(work[feature].dropna()))
    outlier_pct = (z > 3).mean() * 100
    dist_rows.append({"feature": feature, "outlier_pct_3sigma": outlier_pct, "pass_dist": outlier_pct < 1.0})
dist_df = pd.DataFrame(dist_rows)
dist_df

## Summary checklist (from notes)

| Criterion | Rule |
|-----------|------|
| Predictive power | Spearman IC vs `log_return_4h`: \|ρ\| > 0.05, p < 0.05. `IC_1d` / `IC_1w` / `IC_1m` are profile checks only |
| Stability | Rolling IC, IC IR > 0.5 |
| Non-redundant | \|ρ\| with other features < 0.7 |
| Clean distribution | < 1% beyond 3σ |

Flags are **recorded for every feature**. Nothing is removed because a single column fails a screen. `FINAL_FEATURES` is the full research list.

**Next step for strategy work:** feed `FINAL_FEATURES` into a `Strategies/…` class (signals on **shifted** indicators to avoid look-ahead, per notes) and backtest on this READY file.

In [ ]:
redundant_features = {name for pair in redundant_pairs for name in (pair["feature_a"], pair["feature_b"])}
checklist = ic_table.set_index("feature").loc[FINAL_FEATURES, ["IC_4h", "p_4h", "pass_IC"]].copy()
checklist = checklist.join(stability_df.set_index("feature")[["mean_IC", "IC_IR", "pass_stability"]])
checklist["redundant"] = checklist.index.isin(redundant_features)
checklist = checklist.join(dist_df.set_index("feature")[["outlier_pct_3sigma", "pass_dist"]])
checklist